In [3]:
import os, glob, time, random, copy, contextlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, precision_recall_curve)
from torch.optim import AdamW
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# FIXED FOR NUMERICAL STABILITY: Force FP32
# =============================================================================
USE_AMP = False

def autocast_ctx():
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return contextlib.nullcontext()

# =============================================================================
# Configuration
# =============================================================================
@dataclass
class ExperimentConfig:
    hidden: int = 128;  emb_dim: int = 64;  dropout: float = 0.4;  head_dropout: float = 0.3
    n_clients: int = 4;  test_ratio: float = 0.3
    
    # RUNTIME OPTIMIZATIONS
    global_rounds: int = 60;  head_finetune_rounds: int = 20
    ssl_pretrain_rounds: int = 6;  ssl_epochs: int = 20;  sup_epochs: int = 5
    lr: float = 0.001;  lr_min: float = 0.0001
    
    mu_encoder: float = 0.01;  tau: float = 0.7;  feat_drop: float = 0.3
    lam_max: float = 0.6;  lam_warmup_rounds: int = 2;  ema_momentum: float = 0.85
    use_ssl: bool = True;  use_protos: bool = True;  use_fedprox: bool = True
    contrib_floor: float = 0.1;  ece_bins: int = 15

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# =============================================================================
# Data
# =============================================================================
def load_and_build_graph(data_dir, card_time_window=1):
    print("Loading IEEE-CIS Data...")
    txn = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
    idn = pd.read_csv(os.path.join(data_dir, 'train_identity.csv'))
    df = pd.merge(txn, idn, on='TransactionID', how='left')
    y = df['isFraud'].values

    cat_cols = list(set(list(df.select_dtypes(include=['object']).columns) +
                        [c for c in ['card1','card2','card3','card4','card5','card6',
                                     'addr1','addr2','P_emaildomain','R_emaildomain'] if c in df.columns]))
    drop_cols = ['isFraud', 'TransactionID']
    num_cols = [c for c in df.columns if c not in drop_cols and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        df[col] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

    miss_cols = []
    for col in num_cols:
        if df[col].isna().sum() > 0:
            ic = f'{col}__isna'; df[ic] = df[col].isna().astype(np.float32); miss_cols.append(ic)
            df[col] = df[col].fillna(df[col].median())

    if 'TransactionAmt' in df.columns:
        df['TransactionAmt'] = np.log1p(df['TransactionAmt'].clip(lower=0))

    agg_cols = []
    for ac in ['TransactionAmt', 'TransactionDT']:
        if ac in df.columns:
            grp = df.groupby('card1')[ac]
            mc, sc, cc = f'{ac}_c1_mean', f'{ac}_c1_std', f'{ac}_c1_count'
            df[mc] = grp.transform('mean'); df[sc] = grp.transform('std').fillna(0); df[cc] = grp.transform('count')
            agg_cols.extend([mc, sc, cc])
    if 'TransactionAmt' in df.columns and 'TransactionAmt_c1_mean' in df.columns:
        df['Amt_c1_dev'] = df['TransactionAmt'] - df['TransactionAmt_c1_mean']; agg_cols.append('Amt_c1_dev')

    feat_cols = [c for c in num_cols if c != 'TransactionDT'] + cat_cols + miss_cols + agg_cols
    X = np.nan_to_num(StandardScaler().fit_transform(df[feat_cols].values.astype(np.float32)))

    df['orig_idx'] = np.arange(len(df))
    src, dst, edge_dt = [], [], []
    for _, g in df.sort_values(['card1','TransactionDT']).groupby('card1'):
        idxs, times = g['orig_idx'].values, g['TransactionDT'].values
        if len(idxs) < 2: continue
        for off in range(1, min(card_time_window, len(idxs)-1)+1):
            a, b, dt = idxs[:-off], idxs[off:], np.abs(times[off:]-times[:-off]).astype(np.float32)
            src.extend(a); dst.extend(b); edge_dt.extend(dt)
            src.extend(b); dst.extend(a); edge_dt.extend(dt)

    data = Data(x=torch.tensor(X, dtype=torch.float),
                edge_index=torch.tensor([src,dst], dtype=torch.long),
                edge_attr=torch.tensor(np.log1p(np.array(edge_dt, dtype=np.float32))).unsqueeze(-1),
                y=torch.tensor(y, dtype=torch.long),
                timestep=torch.tensor(df['TransactionDT'].values, dtype=torch.long))
    print(f"Graph: {data.num_nodes} nodes | {data.num_edges} edges | {data.num_node_features} feats")
    return data

def stratified_tvt_split(y, seed=42):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, train_size=0.6, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, train_size=0.5, stratify=y[tmp], random_state=seed)
    def mk(i,n): m=torch.zeros(n,dtype=torch.bool); m[i]=True; return m
    return mk(tr,len(y)), mk(va,len(y)), mk(te,len(y))

def temporal_client_split(data, gtm, nc=4, tr=0.3):
    labels, ts = data.y.cpu().numpy(), data.timestep.cpu().numpy()
    ti = np.where(gtm.cpu().numpy())[0]
    splits = np.array_split(ti[np.argsort(ts[ti])], nc)
    clients = []
    for i, sp in enumerate(splits):
        ill, lic = sp[labels[sp]==1], sp[labels[sp]==0]
        def ss(a):
            if len(a)==0: return a,a
            n=max(1,int(len(a)*tr)); return a[:-n],a[-n:]
        it,ie = ss(ill); lt,le = ss(lic)
        t,e = np.concatenate([it,lt]), np.concatenate([ie,le])
        mt,me = torch.zeros(data.num_nodes,dtype=torch.bool), torch.zeros(data.num_nodes,dtype=torch.bool)
        mt[t]=True; me[e]=True
        clients.append({'id':i,'train_mask':mt,'test_mask':me,'n_train':len(t),'n_test':len(e)})
    return clients

def get_local_ei(data, mask, dev):
    ei = data.edge_index.to(dev); m = mask.to(dev); k = m[ei[0]] & m[ei[1]]
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

def get_inductive_ei(data, tmask, temask, dev):
    ei,tr,te = data.edge_index.to(dev), tmask.to(dev), temask.to(dev)
    k = (te[ei[1]] & (tr|te)[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

# =============================================================================
# Model
# =============================================================================
class SAGEGATEncoder(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.feature_proj = nn.Linear(in_dim, h)
        self.conv1 = SAGEConv(h, h); self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False, edge_dim=edge_dim)
        self.skip1 = nn.Linear(h, h, bias=False); self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1 = nn.BatchNorm1d(h); self.bn2 = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(nn.Linear(e, e*2), nn.ReLU(), nn.Linear(e*2, e))

    def _sage(self, x, ei):
        xp = self.feature_proj(x)
        h1 = F.dropout(F.relu(self.bn1(self.conv1(xp,ei)+self.skip1(xp))), self.dropout, self.training)
        h2 = F.dropout(F.relu(self.bn2(self.conv2(h1,ei)+self.skip2(h1))), self.dropout, self.training)
        return h2

    def forward(self, x, ei, ea=None): return self.conv3(self._sage(x,ei), ei, edge_attr=ea)
    def encode_with_proj(self, x, ei, ea=None):
        z = self.forward(x, ei, ea); return z, self.proj_head(z)

class ClassHead(nn.Module):
    def __init__(self, in_dim, cfg):
        super().__init__()
        h1,h2 = max(128,in_dim*2), max(64,in_dim)
        self.net = nn.Sequential(nn.Linear(in_dim,h1),nn.BatchNorm1d(h1),nn.ReLU(),nn.Dropout(cfg.head_dropout),
                                 nn.Linear(h1,h2),nn.ReLU(),nn.Dropout(cfg.head_dropout/2),nn.Linear(h2,2))
    def forward(self,x): return self.net(x)

class FullGAT(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg, edge_dim=edge_dim)
        self.head = ClassHead(cfg.emb_dim, cfg)
    def forward(self, x, ei, ea=None):
        emb = self.encoder(x, ei, ea)
        return self.head(emb), emb

# =============================================================================
# Losses & Federation
# =============================================================================
def supervised_loss(logits, labels, device, gamma=1.0):
    counts = torch.bincount(labels, minlength=2).float().clamp(1.0)
    alpha = (labels.shape[0] / (2*counts)).clamp(0.1, 10.0)
    ce = F.cross_entropy(logits, labels, reduction='none')
    pt = torch.exp(-ce).clamp(1e-7, 1-1e-7)
    return (alpha[labels].to(device) * (1-pt)**gamma * ce).mean()

def compute_protos(emb, data, mask, dev):
    labels, labeled = data.y.to(dev), mask.to(dev) & (data.y>=0)
    return {c: emb[labeled & (labels==c)].mean(0) if (labeled & (labels==c)).sum()>0
            else torch.zeros(emb.shape[1], device=dev) for c in [0,1]}

def proto_supcon(z, labels, mask, gprotos, dev, tau=0.7):
    labeled = mask.to(dev) & (labels.to(dev)>=0)
    if not labeled.any() or gprotos is None: return torch.tensor(0.0, device=dev)
    z_lab = F.normalize(z[labeled].float(), dim=1, eps=1e-5)
    y_lab = labels.to(dev)[labeled]
    p_cat = torch.cat([F.normalize(gprotos[0].float().unsqueeze(0), dim=1, eps=1e-5),
                       F.normalize(gprotos[1].float().unsqueeze(0), dim=1, eps=1e-5)], dim=0)
    return F.cross_entropy(torch.mm(z_lab, p_cat.T) / tau, y_lab)

def multi_budget_proto(z, labels, mask, gprotos, dev, tau, budgets):
    if gprotos is None: return torch.tensor(0.0, device=dev)
    w = 1.0/len(budgets)
    return sum(w * proto_supcon(z[:,:d], labels, mask, {c:v[:d] for c,v in gprotos.items()}, dev, tau) for d in budgets)

def chain_protos(client_ps, cfg):
    if not client_ps: return None
    out = {}
    for i, own in enumerate(client_ps):
        if i == 0:
            out[i] = {c: F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0) for c in [0,1]}
        else:
            pred = client_ps[i-1]
            bl = {}
            for c in [0,1]:
                p = F.normalize(pred[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                o = F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                al = cfg.ema_momentum if c==1 else 0.3
                bl[c] = F.normalize((al*p + (1-al)*o).unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
            out[i] = bl
    return out

def fedavg(gm, lms, sizes):
    excl = ['running_mean','running_var','num_batches_tracked']
    
    valid_lms, valid_sizes = [], []
    for i, lm in enumerate(lms):
        if all(torch.isfinite(p).all() for p in lm.parameters()):
            valid_lms.append(lm)
            valid_sizes.append(sizes[i])
        else:
            print(f"      [WARNING] Client {i} diverged to NaNs! Dropping from aggregation.")
            
    if not valid_lms:
        print("      [FATAL] All clients diverged to NaNs this round!")
        return

    tot = sum(valid_sizes); ns = {}
    for k,v in gm.state_dict().items():
        if any(e in k for e in excl): ns[k]=v; continue
        acc = torch.zeros_like(v.float())
        for i,lm in enumerate(valid_lms): 
            acc += lm.state_dict()[k].float() * (valid_sizes[i]/tot)
        ns[k] = acc.to(v.dtype)
    gm.load_state_dict(ns)

def rank_contrib_agg(cprotos, ctiers, sizes, prev, cfg, tdims):
    bounds = [0]+list(tdims); seg = {0:[], 1:[]}
    for lo,hi in zip(bounds[:-1], bounds[1:]):
        elig = [i for i in range(len(cprotos)) if ctiers[i]>=hi] or [i for i in range(len(cprotos)) if ctiers[i]==max(ctiers)]
        ws = []
        for i in elig:
            if prev is None: ws.append(sizes[i]**0.5); continue
            ls,gs = cprotos[i][1][lo:hi].float(), prev[1][lo:hi].float()
            q = 1.0 if ls.norm()<1e-8 or gs.norm()<1e-8 else cfg.contrib_floor+(1-cfg.contrib_floor)*(F.cosine_similarity(ls.unsqueeze(0),gs.unsqueeze(0),eps=1e-5).item()+1)/2
            ws.append((sizes[i]**0.5)*q)
        tw = sum(ws)+1e-8
        for c in [0,1]:
            acc = torch.zeros(hi-lo, device=cprotos[0][c].device)
            for i,w in zip(elig,ws): acc += (w/tw)*cprotos[i][c][lo:hi]
            seg[c].append(acc)
    return {c: torch.cat(seg[c], dim=0) for c in [0,1]}

# =============================================================================
# Evaluation
# =============================================================================
def evaluate_tuned(model, data, tmask, temask, dev):
    model.eval()
    y_cpu = data.y.cpu()
    zeros = np.zeros(max(1, temask.cpu().sum().item()))
    FAIL = {'f1':0.,'auc':0.,'prec':0.,'rec':0.,'probs':zeros,'true':zeros}

    ei_tr, ea_tr = get_local_ei(data, tmask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_tr, _ = model(data.x.to(dev), ei_tr, ea_tr)
    probs_tr = torch.softmax(logits_tr, dim=1)[:,1].float().cpu().numpy()
    
    if not np.isfinite(probs_tr).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs!")
        return FAIL

    tr_lab = tmask.cpu() & (y_cpu>=0); best_thresh = 0.5
    if tr_lab.sum()>0 and len(np.unique(y_cpu[tr_lab].numpy()))>1:
        p,r,th = precision_recall_curve(y_cpu[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2*p*r/(p+r+1e-8)
        if len(th): best_thresh = float(np.clip(th[np.argmax(f1s[:-1])], 0.05, 0.95))

    ei_te, ea_te = get_inductive_ei(data, tmask, temask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_te, _ = model(data.x.to(dev), ei_te, ea_te)
    probs_te = torch.softmax(logits_te, dim=1)[:,1].float().cpu().numpy()
    
    if not np.isfinite(probs_te).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs on test set!")
        return FAIL

    te_lab = temask.cpu() & (y_cpu>=0)
    if te_lab.sum()==0: return FAIL
    pm, tt = probs_te[te_lab.numpy()], y_cpu[te_lab].numpy()
    pd_ = (pm >= best_thresh).astype(int)
    return {'f1': f1_score(tt,pd_,zero_division=0),
            'auc': roc_auc_score(tt,pm) if len(np.unique(tt))>1 else 0.,
            'prec': precision_score(tt,pd_,zero_division=0),
            'rec': recall_score(tt,pd_,zero_division=0), 'probs':pm, 'true':tt}

def compute_ece(probs, true, n_bins=15):
    bins, ece = np.linspace(0,1,n_bins+1), 0.0
    for i in range(n_bins):
        m = (probs>=bins[i]) & (probs<bins[i+1])
        if m.sum()>0: ece += (m.sum()/len(true))*abs(true[m].mean()-probs[m].mean())
    return float(ece)

def avg_metrics(ml):
    out = {}
    for k in ['f1','auc','prec','rec']:
        vs = [m[k] for m in ml if k in m]
        out[k], out[k+'_std'] = float(np.mean(vs)), float(np.std(vs))
    return out

# =============================================================================
# Shared AMP training step
# =============================================================================
def amp_step(model, opt, scaler, loss):
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(opt); scaler.update()

# =============================================================================
# Method 1: LocalOnly
# =============================================================================
def run_local_only(data, clients, m_val, m_te, dev, cfg, seed):
    set_seed(seed)
    all_m = []
    for ci, c in enumerate(clients):
        model = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
        opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
        ei, ea = get_local_ei(data, c['train_mask'], dev)
        vm = c['train_mask'].to(dev) & (data.y>=0)
        lb = data.y[vm].to(dev)

        best_f1, patience = -float('inf'), 0
        
        # REDUCED LOCAL EPOCHS: 150 instead of 600
        for ep in range(150):
            model.train(); opt.zero_grad(set_to_none=True)
            with autocast_ctx():
                lo, _ = model(data.x.to(dev), ei, ea)
                loss = supervised_loss(lo[vm], lb, dev)
            amp_step(model, opt, scaler, loss)
            
            if (ep+1) % 10 == 0:
                m = evaluate_tuned(model, data, c['train_mask'], m_val, dev)
                if m['f1'] > best_f1: best_f1 = m['f1']; patience = 0
                else: patience += 10
                if patience >= 75: break

        mt = evaluate_tuned(model, data, c['train_mask'], m_te, dev)
        mt['ece'] = compute_ece(mt['probs'], mt['true'])
        all_m.append(mt)
        print(f"    Client {ci}: F1={mt['f1']:.4f} AUC={mt['auc']:.4f}")

    avg = avg_metrics(all_m)
    avg['ece'] = float(np.mean([m['ece'] for m in all_m]))
    avg['comm_bytes'] = 0
    return avg

# =============================================================================
# Method 2: FedProto-8
# =============================================================================
def run_fedproto_8(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    cfg.use_ssl = False; cfg.use_fedprox = False
    PROTO_DIM = 8

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    best_f1, best_state, patience = -float('inf'), copy.deepcopy(gm.state_dict()), 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        # Compute prototypes
        client_ps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            fp = compute_protos(z, data, c['train_mask'], dev)
            client_ps.append({k: v[:PROTO_DIM] for k, v in fp.items()})

        cprotos = chain_protos(client_ps, cfg)
        gprotos = cprotos[0] if cprotos else None

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)

                if gprotos is not None and lam > 0:
                    ploss = proto_supcon(z.float()[:, :PROTO_DIM], data.y,
                                        c['train_mask'], gprotos, dev, cfg.tau)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models 

        # EVALUATE ONLY EVERY 5 ROUNDS
        if (rnd+1) % 5 == 0 or (rnd+1) == cfg.global_rounds:
            m = evaluate_tuned(gm, data, m_tr, m_val, dev)
            if m['f1'] > best_f1: 
                best_f1 = m['f1']
                best_state = copy.deepcopy(gm.state_dict())
                patience = 0
                print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
            else: 
                patience += 1
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
            # Patience 3 = 15 federation rounds of no improvement
            if patience >= 3:
                print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)
    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = n_rounds_run * cfg.n_clients * PROTO_DIM * 2 * 4 * 2
    return mt

# =============================================================================
# Method 3: NEST-64→8
# =============================================================================
def run_nest(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    TIER_DIMS = (8, 16, 32, 64)
    ctiers = [TIER_DIMS[i % len(TIER_DIMS)] for i in range(cfg.n_clients)]
    random.Random(seed).shuffle(ctiers)
    print(f"    Tiers: {ctiers}")

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    prev_protos, best_f1, best_state, patience, total_comm = None, -float('inf'), copy.deepcopy(gm.state_dict()), 0, 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_fps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            client_fps.append(compute_protos(z, data, c['train_mask'], dev))

        gprotos = rank_contrib_agg(client_fps, ctiers, sizes, prev_protos, cfg, TIER_DIMS)
        prev_protos = {c: v.detach().clone() for c,v in gprotos.items()}

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            own_b = tuple(d for d in TIER_DIMS if d <= ctiers[i])
            enc_ref = {n:p.detach().clone() for n,p in gm.encoder.named_parameters()}

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)
                    if cfg.mu_encoder > 0:
                        prox = sum(((p-enc_ref[n])**2).sum() for n,p in lm.encoder.named_parameters())
                        loss = loss + (cfg.mu_encoder/2)*prox

                if lam > 0:
                    ploss = multi_budget_proto(z.float(), data.y, c['train_mask'],
                                              gprotos, dev, cfg.tau, own_b)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models
        total_comm += sum(d*2*4*2 for d in ctiers)

        # EVALUATE ONLY EVERY 5 ROUNDS
        if (rnd+1) % 5 == 0 or (rnd+1) == cfg.global_rounds:
            m = evaluate_tuned(gm, data, m_tr, m_val, dev)
            if m['f1'] > best_f1: 
                best_f1 = m['f1']
                best_state = copy.deepcopy(gm.state_dict())
                patience = 0
                print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
            else: 
                patience += 1
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
            if patience >= 3:
                print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)

    # FedPer head finetune
    for p in gm.encoder.parameters(): p.requires_grad_(False)
    for ft_rnd in range(cfg.head_finetune_rounds):
        for c in clients:
            opt = AdamW(gm.head.parameters(), lr=cfg.lr_min*5)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            for _ in range(5):
                gm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    lo, _ = gm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(lo[vm], lb, dev)
                amp_step(gm, opt, scaler, loss)
    for p in gm.encoder.parameters(): p.requires_grad_(True)

    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = total_comm
    return mt

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    DATA_DIR = '/kaggle/input/competitions/ieee-fraud-detection'
    data = load_and_build_graph(DATA_DIR).to(DEVICE)
    m_tr, m_val, m_te = stratified_tvt_split(data.y.cpu().numpy(), seed=42)
    clients = temporal_client_split(data, m_tr)

    cfg = ExperimentConfig()
    results = {'LocalOnly': [], 'FedProto-8': [], 'NEST-64->8': []}

    for seed in [42, 43, 44]:
        print(f"\n{'='*25} SEED {seed} {'='*25}")

        print("  [LocalOnly]")
        r = run_local_only(data, clients, m_val, m_te, DEVICE, cfg, seed)
        results['LocalOnly'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}")

        print("  [FedProto-8]")
        r = run_fedproto_8(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['FedProto-8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

        print("  [NEST-64->8]")
        r = run_nest(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['NEST-64->8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

    print(f"\n{'='*25} IEEE-CIS External Validation (3-seed) {'='*25}")
    print(f"{'Method':<15} | {'F1 (mean±std)':<18} | {'AUC (mean±std)':<18} | {'Prec':<7} | {'Rec':<7} | {'ECE':<7} | {'Comm(KB)':<9}")
    print("-"*95)
    for method, runs in results.items():
        f1m,f1s = np.mean([r['f1'] for r in runs]), np.std([r['f1'] for r in runs])
        am,as_ = np.mean([r['auc'] for r in runs]), np.std([r['auc'] for r in runs])
        pm,rm = np.mean([r['prec'] for r in runs]), np.mean([r['rec'] for r in runs])
        ec = np.mean([r['ece'] for r in runs])
        cm = np.mean([r['comm_bytes'] for r in runs])/1024
        print(f"{method:<15} | {f1m:.4f} ± {f1s:.4f}   | {am:.4f} ± {as_:.4f}   | {pm:.4f}  | {rm:.4f}  | {ec:.4f}  | {cm:<9.1f}")

Loading IEEE-CIS Data...
Graph: 590540 nodes | 1180046 edges | 817 feats

========================= SEED 42 =========================
  [LocalOnly]
    Client 0: F1=0.3810 AUC=0.7732
    Client 1: F1=0.3343 AUC=0.7915
    Client 2: F1=0.3269 AUC=0.8075
    Client 3: F1=0.3244 AUC=0.8013
  => F1: 0.3417, AUC: 0.7934
  [FedProto-8]
    [NEW BEST] Round   5: F1=0.3537 AUC=0.7933
    [NEW BEST] Round  10: F1=0.3859 AUC=0.8125
    [NEW BEST] Round  15: F1=0.4149 AUC=0.8310
    Round  20: F1=0.4059 AUC=0.8292 (best=0.4149, pat=1)
    [NEW BEST] Round  25: F1=0.4194 AUC=0.8394
    [NEW BEST] Round  30: F1=0.4218 AUC=0.8418
    [NEW BEST] Round  35: F1=0.4240 AUC=0.8427
    [NEW BEST] Round  40: F1=0.4333 AUC=0.8540
    [NEW BEST] Round  45: F1=0.4345 AUC=0.8576
    [NEW BEST] Round  50: F1=0.4402 AUC=0.8586
    Round  55: F1=0.4263 AUC=0.8577 (best=0.4402, pat=1)
    [NEW BEST] Round  10: F1=0.3740 AUC=0.7961
    [NEW BEST] Round  15: F1=0.3990 AUC=0.8135
    [NEW BEST] Round  20: F1=0.4060 A

In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.6 MB/s eta 0:00:0000:01


In [ ]:
import os, glob, time, random, copy, contextlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, precision_recall_curve)
from torch.optim import AdamW
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = False

def autocast_ctx():
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return contextlib.nullcontext()

# =============================================================================
# Configuration
# =============================================================================
@dataclass
class ExperimentConfig:
    hidden: int = 128;  emb_dim: int = 64;  dropout: float = 0.4;  head_dropout: float = 0.3
    n_clients: int = 4;  test_ratio: float = 0.3
    
    global_rounds: int = 60;  head_finetune_rounds: int = 20
    ssl_pretrain_rounds: int = 6;  ssl_epochs: int = 20;  sup_epochs: int = 5
    lr: float = 0.001;  lr_min: float = 0.0001
    
    mu_encoder: float = 0.01;  tau: float = 0.7;  feat_drop: float = 0.3
    # CHANGE 2: Lowered lam_max from 0.6 to 0.2
    lam_max: float = 0.2;  lam_warmup_rounds: int = 2;  ema_momentum: float = 0.85
    
    use_ssl: bool = True;  use_protos: bool = True;  use_fedprox: bool = True
    contrib_floor: float = 0.1;  ece_bins: int = 15

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# =============================================================================
# Data
# =============================================================================
def load_and_build_graph(data_dir, card_time_window=1):
    print("Loading IEEE-CIS Data...")
    txn = pd.read_csv(os.path.join(data_dir, 'train_transaction.csv'))
    idn = pd.read_csv(os.path.join(data_dir, 'train_identity.csv'))
    df = pd.merge(txn, idn, on='TransactionID', how='left')
    y = df['isFraud'].values

    cat_cols = list(set(list(df.select_dtypes(include=['object']).columns) +
                        [c for c in ['card1','card2','card3','card4','card5','card6',
                                     'addr1','addr2','P_emaildomain','R_emaildomain'] if c in df.columns]))
    drop_cols = ['isFraud', 'TransactionID']
    num_cols = [c for c in df.columns if c not in drop_cols and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    for col in cat_cols:
        df[col] = df[col].fillna('__missing__')
        df[col] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)

    miss_cols = []
    for col in num_cols:
        if df[col].isna().sum() > 0:
            ic = f'{col}__isna'; df[ic] = df[col].isna().astype(np.float32); miss_cols.append(ic)
            df[col] = df[col].fillna(df[col].median())

    if 'TransactionAmt' in df.columns:
        df['TransactionAmt'] = np.log1p(df['TransactionAmt'].clip(lower=0))

    agg_cols = []
    for ac in ['TransactionAmt', 'TransactionDT']:
        if ac in df.columns:
            grp = df.groupby('card1')[ac]
            mc, sc, cc = f'{ac}_c1_mean', f'{ac}_c1_std', f'{ac}_c1_count'
            df[mc] = grp.transform('mean'); df[sc] = grp.transform('std').fillna(0); df[cc] = grp.transform('count')
            agg_cols.extend([mc, sc, cc])
    if 'TransactionAmt' in df.columns and 'TransactionAmt_c1_mean' in df.columns:
        df['Amt_c1_dev'] = df['TransactionAmt'] - df['TransactionAmt_c1_mean']; agg_cols.append('Amt_c1_dev')

    feat_cols = [c for c in num_cols if c != 'TransactionDT'] + cat_cols + miss_cols + agg_cols
    X = np.nan_to_num(StandardScaler().fit_transform(df[feat_cols].values.astype(np.float32)))

    df['orig_idx'] = np.arange(len(df))
    src, dst, edge_dt = [], [], []
    for _, g in df.sort_values(['card1','TransactionDT']).groupby('card1'):
        idxs, times = g['orig_idx'].values, g['TransactionDT'].values
        if len(idxs) < 2: continue
        for off in range(1, min(card_time_window, len(idxs)-1)+1):
            a, b, dt = idxs[:-off], idxs[off:], np.abs(times[off:]-times[:-off]).astype(np.float32)
            src.extend(a); dst.extend(b); edge_dt.extend(dt)
            src.extend(b); dst.extend(a); edge_dt.extend(dt)

    data = Data(x=torch.tensor(X, dtype=torch.float),
                edge_index=torch.tensor([src,dst], dtype=torch.long),
                edge_attr=torch.tensor(np.log1p(np.array(edge_dt, dtype=np.float32))).unsqueeze(-1),
                y=torch.tensor(y, dtype=torch.long),
                timestep=torch.tensor(df['TransactionDT'].values, dtype=torch.long))
    print(f"Graph: {data.num_nodes} nodes | {data.num_edges} edges | {data.num_node_features} feats")
    return data

def stratified_tvt_split(y, seed=42):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, train_size=0.6, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, train_size=0.5, stratify=y[tmp], random_state=seed)
    def mk(i,n): m=torch.zeros(n,dtype=torch.bool); m[i]=True; return m
    return mk(tr,len(y)), mk(va,len(y)), mk(te,len(y))

def temporal_client_split(data, gtm, nc=4, tr=0.3):
    labels, ts = data.y.cpu().numpy(), data.timestep.cpu().numpy()
    ti = np.where(gtm.cpu().numpy())[0]
    splits = np.array_split(ti[np.argsort(ts[ti])], nc)
    clients = []
    for i, sp in enumerate(splits):
        ill, lic = sp[labels[sp]==1], sp[labels[sp]==0]
        def ss(a):
            if len(a)==0: return a,a
            n=max(1,int(len(a)*tr)); return a[:-n],a[-n:]
        it,ie = ss(ill); lt,le = ss(lic)
        t,e = np.concatenate([it,lt]), np.concatenate([ie,le])
        mt,me = torch.zeros(data.num_nodes,dtype=torch.bool), torch.zeros(data.num_nodes,dtype=torch.bool)
        mt[t]=True; me[e]=True
        clients.append({'id':i,'train_mask':mt,'test_mask':me,'n_train':len(t),'n_test':len(e)})
    return clients

def get_local_ei(data, mask, dev):
    ei = data.edge_index.to(dev); m = mask.to(dev); k = m[ei[0]] & m[ei[1]]
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

def get_inductive_ei(data, tmask, temask, dev):
    ei,tr,te = data.edge_index.to(dev), tmask.to(dev), temask.to(dev)
    k = (te[ei[1]] & (tr|te)[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:,k], (data.edge_attr.to(dev)[k] if data.edge_attr is not None else None)

# =============================================================================
# Model
# =============================================================================
class SAGEGATEncoder(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        h, e, dr = cfg.hidden, cfg.emb_dim, cfg.dropout
        self.dropout = dr
        self.feature_proj = nn.Linear(in_dim, h)
        self.conv1 = SAGEConv(h, h); self.conv2 = SAGEConv(h, h)
        self.conv3 = GATv2Conv(h, e, heads=1, dropout=dr, concat=False, edge_dim=edge_dim)
        self.skip1 = nn.Linear(h, h, bias=False); self.skip2 = nn.Linear(h, h, bias=False)
        self.bn1 = nn.BatchNorm1d(h); self.bn2 = nn.BatchNorm1d(h)
        self.proj_head = nn.Sequential(nn.Linear(e, e*2), nn.ReLU(), nn.Linear(e*2, e))

    def _sage(self, x, ei):
        xp = self.feature_proj(x)
        h1 = F.dropout(F.relu(self.bn1(self.conv1(xp,ei)+self.skip1(xp))), self.dropout, self.training)
        h2 = F.dropout(F.relu(self.bn2(self.conv2(h1,ei)+self.skip2(h1))), self.dropout, self.training)
        return h2

    def forward(self, x, ei, ea=None): return self.conv3(self._sage(x,ei), ei, edge_attr=ea)
    def encode_with_proj(self, x, ei, ea=None):
        z = self.forward(x, ei, ea); return z, self.proj_head(z)

class ClassHead(nn.Module):
    def __init__(self, in_dim, cfg):
        super().__init__()
        h1,h2 = max(128,in_dim*2), max(64,in_dim)
        self.net = nn.Sequential(nn.Linear(in_dim,h1),nn.BatchNorm1d(h1),nn.ReLU(),nn.Dropout(cfg.head_dropout),
                                 nn.Linear(h1,h2),nn.ReLU(),nn.Dropout(cfg.head_dropout/2),nn.Linear(h2,2))
    def forward(self,x): return self.net(x)

class FullGAT(nn.Module):
    def __init__(self, in_dim, cfg, edge_dim=None):
        super().__init__()
        self.encoder = SAGEGATEncoder(in_dim, cfg, edge_dim=edge_dim)
        self.head = ClassHead(cfg.emb_dim, cfg)
    def forward(self, x, ei, ea=None):
        emb = self.encoder(x, ei, ea)
        return self.head(emb), emb

# =============================================================================
# Losses & Federation
# =============================================================================
def supervised_loss(logits, labels, device, gamma=1.0):
    counts = torch.bincount(labels, minlength=2).float().clamp(1.0)
    alpha = (labels.shape[0] / (2*counts)).clamp(0.1, 10.0)
    ce = F.cross_entropy(logits, labels, reduction='none')
    pt = torch.exp(-ce).clamp(1e-7, 1-1e-7)
    return (alpha[labels].to(device) * (1-pt)**gamma * ce).mean()

def compute_protos(emb, data, mask, dev):
    labels, labeled = data.y.to(dev), mask.to(dev) & (data.y>=0)
    return {c: emb[labeled & (labels==c)].mean(0) if (labeled & (labels==c)).sum()>0
            else torch.zeros(emb.shape[1], device=dev) for c in [0,1]}

def proto_supcon(z, labels, mask, gprotos, dev, tau=0.7):
    labeled = mask.to(dev) & (labels.to(dev)>=0)
    if not labeled.any() or gprotos is None: return torch.tensor(0.0, device=dev)
    z_lab = F.normalize(z[labeled].float(), dim=1, eps=1e-5)
    y_lab = labels.to(dev)[labeled]
    p_cat = torch.cat([F.normalize(gprotos[0].float().unsqueeze(0), dim=1, eps=1e-5),
                       F.normalize(gprotos[1].float().unsqueeze(0), dim=1, eps=1e-5)], dim=0)
    return F.cross_entropy(torch.mm(z_lab, p_cat.T) / tau, y_lab)

# CHANGE 3: Progressively weighted budgets
def multi_budget_proto(z, labels, mask, gprotos, dev, tau, budgets):
    if gprotos is None: return torch.tensor(0.0, device=dev)
    
    weights = torch.tensor([1.0, 0.5, 0.25, 0.125][:len(budgets)], device=dev)
    weights = weights / weights.sum()
    
    loss = 0.0
    for w, d in zip(weights, budgets):
        loss = loss + w * proto_supcon(z[:, :d], labels, mask, {c: v[:d] for c, v in gprotos.items()}, dev, tau)
        
    return loss

def chain_protos(client_ps, cfg):
    if not client_ps: return None
    out = {}
    for i, own in enumerate(client_ps):
        if i == 0:
            out[i] = {c: F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0) for c in [0,1]}
        else:
            pred = client_ps[i-1]
            bl = {}
            for c in [0,1]:
                p = F.normalize(pred[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                o = F.normalize(own[c].float().unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
                al = cfg.ema_momentum if c==1 else 0.3
                bl[c] = F.normalize((al*p + (1-al)*o).unsqueeze(0), dim=1, eps=1e-5).squeeze(0)
            out[i] = bl
    return out

def fedavg(gm, lms, sizes):
    excl = ['running_mean','running_var','num_batches_tracked']
    
    valid_lms, valid_sizes = [], []
    for i, lm in enumerate(lms):
        if all(torch.isfinite(p).all() for p in lm.parameters()):
            valid_lms.append(lm)
            valid_sizes.append(sizes[i])
        else:
            print(f"      [WARNING] Client {i} diverged to NaNs! Dropping from aggregation.")
            
    if not valid_lms:
        print("      [FATAL] All clients diverged to NaNs this round!")
        return

    tot = sum(valid_sizes); ns = {}
    for k,v in gm.state_dict().items():
        if any(e in k for e in excl): ns[k]=v; continue
        acc = torch.zeros_like(v.float())
        for i,lm in enumerate(valid_lms): 
            acc += lm.state_dict()[k].float() * (valid_sizes[i]/tot)
        ns[k] = acc.to(v.dtype)
    gm.load_state_dict(ns)

def rank_contrib_agg(cprotos, ctiers, sizes, prev, cfg, tdims):
    bounds = [0]+list(tdims); seg = {0:[], 1:[]}
    for lo,hi in zip(bounds[:-1], bounds[1:]):
        elig = [i for i in range(len(cprotos)) if ctiers[i]>=hi] or [i for i in range(len(cprotos)) if ctiers[i]==max(ctiers)]
        ws = []
        for i in elig:
            if prev is None: ws.append(sizes[i]**0.5); continue
            ls,gs = cprotos[i][1][lo:hi].float(), prev[1][lo:hi].float()
            q = 1.0 if ls.norm()<1e-8 or gs.norm()<1e-8 else cfg.contrib_floor+(1-cfg.contrib_floor)*(F.cosine_similarity(ls.unsqueeze(0),gs.unsqueeze(0),eps=1e-5).item()+1)/2
            ws.append((sizes[i]**0.5)*q)
            
        # DIAGNOSTIC PRINT for feedback loops
        print(f"      dim {lo}:{hi} | " + " ".join(f"C{i}={w:.3f}" for i,w in zip(elig,ws)))
        
        tw = sum(ws)+1e-8
        for c in [0,1]:
            acc = torch.zeros(hi-lo, device=cprotos[0][c].device)
            for i,w in zip(elig,ws): acc += (w/tw)*cprotos[i][c][lo:hi]
            seg[c].append(acc)
    return {c: torch.cat(seg[c], dim=0) for c in [0,1]}

# =============================================================================
# Evaluation
# =============================================================================
def evaluate_tuned(model, data, tmask, temask, dev):
    model.eval()
    y_cpu = data.y.cpu()
    zeros = np.zeros(max(1, temask.cpu().sum().item()))
    FAIL = {'f1':0.,'auc':0.,'prec':0.,'rec':0.,'probs':zeros,'true':zeros}

    ei_tr, ea_tr = get_local_ei(data, tmask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_tr, _ = model(data.x.to(dev), ei_tr, ea_tr)
    probs_tr = torch.softmax(logits_tr, dim=1)[:,1].float().cpu().numpy()
    
    if not np.isfinite(probs_tr).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs!")
        return FAIL

    tr_lab = tmask.cpu() & (y_cpu>=0); best_thresh = 0.5
    if tr_lab.sum()>0 and len(np.unique(y_cpu[tr_lab].numpy()))>1:
        p,r,th = precision_recall_curve(y_cpu[tr_lab].numpy(), probs_tr[tr_lab.numpy()])
        f1s = 2*p*r/(p+r+1e-8)
        if len(th): best_thresh = float(np.clip(th[np.argmax(f1s[:-1])], 0.05, 0.95))

    ei_te, ea_te = get_inductive_ei(data, tmask, temask, dev)
    with torch.no_grad(), autocast_ctx():
        logits_te, _ = model(data.x.to(dev), ei_te, ea_te)
    probs_te = torch.softmax(logits_te, dim=1)[:,1].float().cpu().numpy()
    
    if not np.isfinite(probs_te).all(): 
        print("      [WARNING] Evaluation aborted: Logits diverged to NaNs on test set!")
        return FAIL

    te_lab = temask.cpu() & (y_cpu>=0)
    if te_lab.sum()==0: return FAIL
    pm, tt = probs_te[te_lab.numpy()], y_cpu[te_lab].numpy()
    pd_ = (pm >= best_thresh).astype(int)
    return {'f1': f1_score(tt,pd_,zero_division=0),
            'auc': roc_auc_score(tt,pm) if len(np.unique(tt))>1 else 0.,
            'prec': precision_score(tt,pd_,zero_division=0),
            'rec': recall_score(tt,pd_,zero_division=0), 'probs':pm, 'true':tt}

def compute_ece(probs, true, n_bins=15):
    bins, ece = np.linspace(0,1,n_bins+1), 0.0
    for i in range(n_bins):
        m = (probs>=bins[i]) & (probs<bins[i+1])
        if m.sum()>0: ece += (m.sum()/len(true))*abs(true[m].mean()-probs[m].mean())
    return float(ece)

def avg_metrics(ml):
    out = {}
    for k in ['f1','auc','prec','rec']:
        vs = [m[k] for m in ml if k in m]
        out[k], out[k+'_std'] = float(np.mean(vs)), float(np.std(vs))
    return out

# =============================================================================
# Shared AMP training step
# =============================================================================
def amp_step(model, opt, scaler, loss):
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(opt); scaler.update()

# =============================================================================
# Method 1: LocalOnly
# =============================================================================
def run_local_only(data, clients, m_val, m_te, dev, cfg, seed):
    set_seed(seed)
    all_m = []
    for ci, c in enumerate(clients):
        model = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
        opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
        ei, ea = get_local_ei(data, c['train_mask'], dev)
        vm = c['train_mask'].to(dev) & (data.y>=0)
        lb = data.y[vm].to(dev)

        best_f1, patience = -float('inf'), 0
        for ep in range(150):
            model.train(); opt.zero_grad(set_to_none=True)
            with autocast_ctx():
                lo, _ = model(data.x.to(dev), ei, ea)
                loss = supervised_loss(lo[vm], lb, dev)
            amp_step(model, opt, scaler, loss)
            
            if (ep+1) % 10 == 0:
                m = evaluate_tuned(model, data, c['train_mask'], m_val, dev)
                if m['f1'] > best_f1: best_f1 = m['f1']; patience = 0
                else: patience += 10
                if patience >= 75: break

        mt = evaluate_tuned(model, data, c['train_mask'], m_te, dev)
        mt['ece'] = compute_ece(mt['probs'], mt['true'])
        all_m.append(mt)
        print(f"    Client {ci}: F1={mt['f1']:.4f} AUC={mt['auc']:.4f}")

    avg = avg_metrics(all_m)
    avg['ece'] = float(np.mean([m['ece'] for m in all_m]))
    avg['comm_bytes'] = 0
    return avg

# =============================================================================
# Method 2: FedProto-8
# =============================================================================
def run_fedproto_8(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    cfg.use_ssl = False; cfg.use_fedprox = False
    PROTO_DIM = 8

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    best_f1, best_state, patience = -float('inf'), copy.deepcopy(gm.state_dict()), 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        # Compute prototypes
        client_ps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            fp = compute_protos(z, data, c['train_mask'], dev)
            client_ps.append({k: v[:PROTO_DIM] for k, v in fp.items()})

        cprotos = chain_protos(client_ps, cfg)
        gprotos = cprotos[0] if cprotos else None

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)

                if gprotos is not None and lam > 0:
                    ploss = proto_supcon(z.float()[:, :PROTO_DIM], data.y,
                                        c['train_mask'], gprotos, dev, cfg.tau)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models 

        if (rnd+1) % 5 == 0 or (rnd+1) == cfg.global_rounds:
            m = evaluate_tuned(gm, data, m_tr, m_val, dev)
            if m['f1'] > best_f1: 
                best_f1 = m['f1']
                best_state = copy.deepcopy(gm.state_dict())
                patience = 0
                print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
            else: 
                patience += 1
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
            if patience >= 3:
                print(f"    Early stop at round {rnd+1}"); break

    gm.load_state_dict(best_state)
    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = n_rounds_run * cfg.n_clients * PROTO_DIM * 2 * 4 * 2
    return mt

# =============================================================================
# Method 3: NEST-64→8
# =============================================================================
def run_nest(data, clients, m_tr, m_val, m_te, dev, base_cfg, seed):
    set_seed(seed)
    cfg = copy.deepcopy(base_cfg)
    TIER_DIMS = (8, 16, 32, 64)
    ctiers = [TIER_DIMS[i % len(TIER_DIMS)] for i in range(cfg.n_clients)]
    random.Random(seed).shuffle(ctiers)
    print(f"    Tiers: {ctiers}")

    gm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
    sizes = [c['n_train'] for c in clients]
    
    prev_protos, best_f1, best_state, patience, total_comm = None, -float('inf'), copy.deepcopy(gm.state_dict()), 0, 0
    n_rounds_run = 0

    for rnd in range(cfg.global_rounds):
        n_rounds_run = rnd + 1
        lam = cfg.lam_max * min(1.0, rnd / max(cfg.lam_warmup_rounds, 1))

        client_fps = []
        for c in clients:
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            with torch.no_grad(), autocast_ctx():
                _, z = gm(data.x.to(dev), ei, ea)
            client_fps.append(compute_protos(z, data, c['train_mask'], dev))

        gprotos = rank_contrib_agg(client_fps, ctiers, sizes, prev_protos, cfg, TIER_DIMS)
        prev_protos = {c: v.detach().clone() for c,v in gprotos.items()}

        local_models = []
        for i, c in enumerate(clients):
            lm = FullGAT(data.num_node_features, cfg, edge_dim=1).to(dev)
            lm.load_state_dict(gm.state_dict())
            opt = AdamW(lm.parameters(), lr=cfg.lr, weight_decay=1e-4)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
            ei, ea = get_local_ei(data, c['train_mask'], dev)
            vm = c['train_mask'].to(dev) & (data.y>=0)
            lb = data.y[vm].to(dev)
            own_b = tuple(d for d in TIER_DIMS if d <= ctiers[i])
            enc_ref = {n:p.detach().clone() for n,p in gm.encoder.named_parameters()}

            for _ in range(cfg.sup_epochs):
                lm.train(); opt.zero_grad(set_to_none=True)
                with autocast_ctx():
                    logits, z = lm(data.x.to(dev), ei, ea)
                    loss = supervised_loss(logits[vm], lb, dev)
                    if cfg.mu_encoder > 0:
                        prox = sum(((p-enc_ref[n])**2).sum() for n,p in lm.encoder.named_parameters())
                        loss = loss + (cfg.mu_encoder/2)*prox

                if lam > 0:
                    ploss = multi_budget_proto(z.float(), data.y, c['train_mask'],
                                              gprotos, dev, cfg.tau, own_b)
                    loss = loss + lam * ploss

                amp_step(lm, opt, scaler, loss)
            local_models.append(lm)

        fedavg(gm, local_models, sizes)
        del local_models
        total_comm += sum(d*2*4*2 for d in ctiers)

        if (rnd+1) % 5 == 0 or (rnd+1) == cfg.global_rounds:
            m = evaluate_tuned(gm, data, m_tr, m_val, dev)
            if m['f1'] > best_f1: 
                best_f1 = m['f1']
                best_state = copy.deepcopy(gm.state_dict())
                patience = 0
                print(f"    [NEW BEST] Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f}")
            else: 
                patience += 1
                print(f"    Round {rnd+1:3d}: F1={m['f1']:.4f} AUC={m['auc']:.4f} (best={best_f1:.4f}, pat={patience})")
                
            if patience >= 3:
                print(f"    Early stop at round {rnd+1}"); break

    # CHANGE 1: Removed FedPer Head Finetuning
    gm.load_state_dict(best_state)

    mt = evaluate_tuned(gm, data, m_tr, m_te, dev)
    mt['ece'] = compute_ece(mt['probs'], mt['true'])
    mt['comm_bytes'] = total_comm
    return mt

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    DATA_DIR = '/kaggle/input/competitions/ieee-fraud-detection'
    data = load_and_build_graph(DATA_DIR).to(DEVICE)
    m_tr, m_val, m_te = stratified_tvt_split(data.y.cpu().numpy(), seed=42)
    clients = temporal_client_split(data, m_tr)

    cfg = ExperimentConfig()
    results = {'LocalOnly': [], 'FedProto-8': [], 'NEST-64->8': []}

    for seed in [42, 43, 44]:
        print(f"\n{'='*25} SEED {seed} {'='*25}")

        print("  [LocalOnly]")
        r = run_local_only(data, clients, m_val, m_te, DEVICE, cfg, seed)
        results['LocalOnly'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}")

        print("  [FedProto-8]")
        r = run_fedproto_8(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['FedProto-8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

        print("  [NEST-64->8]")
        r = run_nest(data, clients, m_tr, m_val, m_te, DEVICE, cfg, seed)
        results['NEST-64->8'].append(r)
        print(f"  => F1: {r['f1']:.4f}, AUC: {r['auc']:.4f}, Comm: {r['comm_bytes']/1024:.1f} KB")

    print(f"\n{'='*25} IEEE-CIS External Validation (3-seed) {'='*25}")
    print(f"{'Method':<15} | {'F1 (mean±std)':<18} | {'AUC (mean±std)':<18} | {'Prec':<7} | {'Rec':<7} | {'ECE':<7} | {'Comm(KB)':<9}")
    print("-"*95)
    for method, runs in results.items():
        f1m,f1s = np.mean([r['f1'] for r in runs]), np.std([r['f1'] for r in runs])
        am,as_ = np.mean([r['auc'] for r in runs]), np.std([r['auc'] for r in runs])
        pm,rm = np.mean([r['prec'] for r in runs]), np.mean([r['rec'] for r in runs])
        ec = np.mean([r['ece'] for r in runs])
        cm = np.mean([r['comm_bytes'] for r in runs])/1024
        print(f"{method:<15} | {f1m:.4f} ± {f1s:.4f}   | {am:.4f} ± {as_:.4f}   | {pm:.4f}  | {rm:.4f}  | {ec:.4f}  | {cm:<9.1f}")

Loading IEEE-CIS Data...
Graph: 590540 nodes | 1180046 edges | 817 feats

========================= SEED 42 =========================
  [LocalOnly]
    Client 0: F1=0.3864 AUC=0.7727
    Client 1: F1=0.3258 AUC=0.7923
    Client 2: F1=0.3328 AUC=0.8074
    Client 3: F1=0.3244 AUC=0.8013
  => F1: 0.3424, AUC: 0.7935
  [FedProto-8]
    [NEW BEST] Round   5: F1=0.3510 AUC=0.7995
    [NEW BEST] Round  10: F1=0.4032 AUC=0.8251
    [NEW BEST] Round  15: F1=0.4145 AUC=0.8383
    [NEW BEST] Round  20: F1=0.4208 AUC=0.8432
    [NEW BEST] Round  25: F1=0.4273 AUC=0.8493
    [NEW BEST] Round  30: F1=0.4293 AUC=0.8565
    [NEW BEST] Round  35: F1=0.4418 AUC=0.8600
    [NEW BEST] Round  40: F1=0.4439 AUC=0.8621
    [NEW BEST] Round  45: F1=0.4475 AUC=0.8596
    [NEW BEST] Round  50: F1=0.4479 AUC=0.8634
    [NEW BEST] Round  55: F1=0.4589 AUC=0.8642
    [NEW BEST] Round  60: F1=0.4631 AUC=0.8614
  => F1: 0.4550, AUC: 0.8606, Comm: 30.0 KB
  [NEST-64->8]
    Tiers: [32, 16, 64, 8]
      dim 0:8 | C0